In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 8.5 MB/s eta 0:00:00


In [2]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [6]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2026-01-14 16:54:41,610] A new study created in memory with name: no-name-e71aa542-9b16-4ff0-ac62-841965d40bbe
[I 2026-01-14 16:54:41,920] Trial 0 finished with value: 0.7802607076350093 and parameters: {'n_estimators': 56, 'max_depth': 7}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-01-14 16:54:42,629] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 123, 'max_depth': 13}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-01-14 16:54:43,016] Trial 2 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 73, 'max_depth': 7}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-01-14 16:54:43,726] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 130, 'max_depth': 11}. Best is trial 0 with value: 0.7802607076350093.
[I 2026-01-14 16:54:44,728] Trial 4 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 184, 'max_depth': 15}. Best is trial 0 with value: 0.78026070

In [7]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 56, 'max_depth': 7}


In [8]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


## Samplers in Optuna

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [10]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-01-14 16:56:39,486] A new study created in memory with name: no-name-33c45770-fe0f-4cff-88fe-f59d3d9b24ae
[I 2026-01-14 16:56:40,020] Trial 0 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 63, 'max_depth': 8}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-01-14 16:56:41,402] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 198, 'max_depth': 7}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-01-14 16:56:42,178] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 142, 'max_depth': 20}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-01-14 16:56:42,753] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 110, 'max_depth': 5}. Best is trial 0 with value: 0.7746741154562384.
[I 2026-01-14 16:56:43,789] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 191, 'max_depth': 18}. Best is trial 0 with value: 0.77467411

In [11]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420856
Best hyperparameters: {'n_estimators': 118, 'max_depth': 19}


In [12]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


In [13]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [14]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-01-14 16:57:35,088] A new study created in memory with name: no-name-2ba3f08f-77e7-424a-a7a7-57d033d3a10c
[I 2026-01-14 16:57:35,592] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-01-14 16:57:36,419] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-01-14 16:57:36,701] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-01-14 16:57:37,274] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-01-14 16:57:37,849] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [15]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [16]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [17]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [18]:
# 1. Optimization History
plot_optimization_history(study).show()

In [19]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [20]:
# 3. Slice Plot
plot_slice(study).show()

In [21]:
# 4. Contour Plot
plot_contour(study).show()

In [22]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [23]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [24]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [25]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-01-14 17:01:29,617] A new study created in memory with name: no-name-29b231b9-9a42-487f-a18e-b0b45ec017cc
[I 2026-01-14 17:01:32,834] Trial 0 finished with value: 0.7374301675977654 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 230, 'learning_rate': 0.11402246735580267, 'max_depth': 19, 'min_samples_split': 2, 'min_samples_leaf': 7}. Best is trial 0 with value: 0.7374301675977654.
[I 2026-01-14 17:01:33,623] Trial 1 finished with value: 0.7653631284916201 and parameters: {'classifier': 'RandomForest', 'n_estimators': 152, 'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 3, 'bootstrap': True}. Best is trial 1 with value: 0.7653631284916201.
[I 2026-01-14 17:01:33,667] Trial 2 finished with value: 0.7243947858472998 and parameters: {'classifier': 'SVM', 'C': 1.8347240186690266, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 1 with value: 0.7653631284916201.
[I 2026-01-14 17:01:34,869] Trial 3 finished with value: 0.7709497206703911 and para

In [26]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.1206071379136735, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [27]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.737430,2026-01-14 17:01:29.620213,2026-01-14 17:01:32.834748,0 days 00:00:03.214535,NaN,NaN,GradientBoosting,NaN,NaN,0.114022,19.0,7.0,2.0,230.0,COMPLETE
1,1,0.765363,2026-01-14 17:01:32.835691,2026-01-14 17:01:33.623819,0 days 00:00:00.788128,NaN,True,RandomForest,NaN,NaN,NaN,15.0,3.0,3.0,152.0,COMPLETE
2,2,0.724395,2026-01-14 17:01:33.624813,2026-01-14 17:01:33.667842,0 days 00:00:00.043029,1.834724,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.770950,2026-01-14 17:01:33.668739,2026-01-14 17:01:34.869140,0 days 00:00:01.200401,NaN,False,RandomForest,NaN,NaN,NaN,11.0,7.0,4.0,275.0,COMPLETE
4,4,0.770950,2026-01-14 17:01:34.870346,2026-01-14 17:01:36.442552,0 days 00:00:01.572206,NaN,False,RandomForest,NaN,NaN,NaN,7.0,2.0,6.0,267.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.787709,2026-01-14 17:02:07.615253,2026-01-14 17:02:07.643165,0 days 00:00:00.027912,0.101459,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2026-01-14 17:02:07.644037,2026-01-14 17:02:07.678590,0 days 00:00:00.034553,1.668951,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.785847,2026-01-14 17:02:07.679432,2026-01-14 17:02:07.706810,0 days 00:00:00.027378,0.205487,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.789572,2026-01-14 17:02:07.707597,2026-01-14 17:02:07.736535,0 days 00:00:00.028938,0.150404,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [28]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,78
RandomForest,12
GradientBoosting,10


In [29]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.747300
RandomForest,0.766760
SVM,0.775868


In [30]:
# 1. Optimization History
plot_optimization_history(study).show()

In [31]:
# 3. Slice Plot
plot_slice(study).show()

In [32]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [35]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")

[I 2026-01-14 17:04:53,976] A new study created in memory with name: no-name-a5b19e5e-e5e3-4eda-968c-0874ef936b8f


[0]	train-mlogloss:1.08659	eval-mlogloss:1.08941
[1]	train-mlogloss:1.05997	eval-mlogloss:1.06149
[2]	train-mlogloss:1.03429	eval-mlogloss:1.03426
[3]	train-mlogloss:1.01501	eval-mlogloss:1.01335
[4]	train-mlogloss:0.99431	eval-mlogloss:0.99251
[5]	train-mlogloss:0.97095	eval-mlogloss:0.96768
[6]	train-mlogloss:0.95331	eval-mlogloss:0.94991
[7]	train-mlogloss:0.93943	eval-mlogloss:0.93638
[8]	train-mlogloss:0.91791	eval-mlogloss:0.91347
[9]	train-mlogloss:0.90168	eval-mlogloss:0.89713
[10]	train-mlogloss:0.88123	eval-mlogloss:0.87589
[11]	train-mlogloss:0.86362	eval-mlogloss:0.85739
[12]	train-mlogloss:0.85510	eval-mlogloss:0.84987
[13]	train-mlogloss:0.83630	eval-mlogloss:0.82959
[14]	train-mlogloss:0.82909	eval-mlogloss:0.82327
[15]	train-mlogloss:0.81797	eval-mlogloss:0.81247
[16]	train-mlogloss:0.80625	eval-mlogloss:0.80052
[17]	train-mlogloss:0.79288	eval-mlogloss:0.78705
[18]	train-mlogloss:0.78245	eval-mlogloss:0.77691
[19]	train-mlogloss:0.77226	eval-mlogloss:0.76713
[20]	train

[I 2026-01-14 17:04:59,985] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.08427273447601352, 'alpha': 0.038391228988558825, 'eta': 0.020729016459737865, 'gamma': 0.029346276607910678, 'max_depth': 4, 'min_child_weight': 4, 'subsample': 0.9973967722018147, 'colsample_bytree': 0.44817879607643973}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.87504	eval-mlogloss:0.88588
[1]	train-mlogloss:0.67317	eval-mlogloss:0.68292
[2]	train-mlogloss:0.52529	eval-mlogloss:0.52582
[3]	train-mlogloss:0.41938	eval-mlogloss:0.41558
[4]	train-mlogloss:0.34039	eval-mlogloss:0.33200
[5]	train-mlogloss:0.27777	eval-mlogloss:0.26039
[6]	train-mlogloss:0.22857	eval-mlogloss:0.22372
[7]	train-mlogloss:0.19390	eval-mlogloss:0.18373
[8]	train-mlogloss:0.16615	eval-mlogloss:0.14963
[9]	train-mlogloss:0.14451	eval-mlogloss:0.12426
[10]	train-mlogloss:0.12513	eval-mlogloss:0.10724
[11]	train-mlogloss:0.11013	eval-mlogloss:0.09017
[12]	train-mlogloss:0.10197	eval-mlogloss:0.08458
[13]	train-mlogloss:0.09046	eval-mlogloss:0.07211
[14]	train-mlogloss:0.08466	eval-mlogloss:0.06750
[15]	train-mlogloss:0.07819	eval-mlogloss:0.05788
[16]	train-mlogloss:0.07428	eval-mlogloss:0.05222
[17]	train-mlogloss:0.06952	eval-mlogloss:0.04935
[18]	train-mlogloss:0.06563	eval-mlogloss:0.04746
[19]	train-mlogloss:0.06296	eval-mlogloss:0.04665
[20]	train

[I 2026-01-14 17:05:02,177] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.1391070035368807, 'alpha': 0.024458242961033126, 'eta': 0.2333336048260406, 'gamma': 8.591197421786614e-05, 'max_depth': 4, 'min_child_weight': 1, 'subsample': 0.7341147074950447, 'colsample_bytree': 0.57341718582612}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.97561	eval-mlogloss:0.98485
[1]	train-mlogloss:0.85008	eval-mlogloss:0.85878


[I 2026-01-14 17:05:19,258] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:0.91177	eval-mlogloss:0.91873
[1]	train-mlogloss:0.71714	eval-mlogloss:0.70990


[I 2026-01-14 17:05:19,299] Trial 49 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 0.08427273447601352, 'alpha': 0.038391228988558825, 'eta': 0.020729016459737865, 'gamma': 0.029346276607910678, 'max_depth': 4, 'min_child_weight': 4, 'subsample': 0.9973967722018147, 'colsample_bytree': 0.44817879607643973}
Best accuracy: 1.0


In [34]:
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 3.0 MB/s eta 0:00:00


In [36]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()